# Pendubot Control Workflow

This notebook is the script-based workflow for `compensation_controller/acados_pendubot_swingup.ipynb`. It uses the latest SysID parameter json for the configured `cell_id`, generates a pendubot reference, and can optionally start the hardware NMPC run.

## Flow

1. Load `sysid_config.json`.
2. Select latest `identified_params_{cell}_fit_*.json`.
3. Generate `pendubot_reference_{cell}.npz` with parameter sha256 provenance.
4. Optionally run `ctrl_run_pendubot.py` on hardware.
5. Show latest `ctrlrun_pendubot_{cell}_*.npz` log.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if (ROOT / "ctrl_run_pendubot.py").exists():
    CTRL_DIR = ROOT
elif (ROOT / "sysid_pipeline" / "ctrl_run_pendubot.py").exists():
    CTRL_DIR = ROOT / "sysid_pipeline"
else:
    raise FileNotFoundError("Run this notebook from sysid_pipeline or the project root.")

print(f"Control root: {CTRL_DIR}")

Control root: /home/jupyter-venky/underactuated_double_pendulum_package


## Settings

In [2]:
PLANT = "pendubot"

# Generate a fresh reference before hardware control.
MAKE_REFERENCE = True

# Hardware run is off by default. Set True only when the cell is ready.
RUN_HARDWARE = False

T_SIM = 20.0
RECORD = True

# If empty, use the latest identified_params_{cell}_fit_*.json.
PARAM_FILE = None

# Advanced options.
# Leave None to use ACADOS_SOURCE_DIR/ACADOS_ROOT/~/acados.
ACADOS_ROOT = None
ALLOW_IPOPT_FALLBACK = False
ALLOW_LEGACY_REF = False

## Helpers

In [3]:
def run_step(tag, args, check=True):
    cmd = [sys.executable, *map(str, args)]
    print("\n" + "=" * 72)
    print(f"{tag}: {' '.join([Path(sys.executable).name, *map(str, args)])}")
    print("=" * 72, flush=True)
    env = os.environ.copy()
    env.setdefault("PYTHONIOENCODING", "utf-8")
    if globals().get("ACADOS_ROOT"):
        acados_root = str(Path(ACADOS_ROOT).expanduser().resolve())
        env["ACADOS_SOURCE_DIR"] = acados_root
        env["ACADOS_ROOT"] = acados_root
    proc = subprocess.Popen(
        cmd,
        cwd=str(CTRL_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print(f"\nexit={rc}")
    if check and rc != 0:
        raise RuntimeError(f"{tag} failed with exit code {rc}")
    return rc


def load_config():
    with (CTRL_DIR / "sysid_config.json").open("r", encoding="utf-8") as f:
        return json.load(f)


def data_root(cfg):
    return (CTRL_DIR / cfg.get("data_dir", ".")).resolve()


def latest_params(cfg):
    if PARAM_FILE:
        p = Path(PARAM_FILE)
        return p if p.is_absolute() else data_root(cfg) / p
    root = data_root(cfg)
    cell = cfg["cell_id"]
    files = sorted(root.glob(f"identified_params_{cell}_fit_*.json"), key=lambda p: p.stat().st_mtime)
    if not files:
        files = sorted(root.glob(f"identified_params_{cell}_*.json"), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError("No identified parameter json found. Run sysid_workflow.ipynb first.")
    return files[-1]


cfg = load_config()
PARAM_PATH = latest_params(cfg)
print(f"cell_id: {cfg['cell_id']}")
print(f"params : {PARAM_PATH.name}")

cell_id: 202
params : identified_params_202_fit_20260721-231023.json


## Generate Reference

In [4]:
if MAKE_REFERENCE:
    run_step(
        "Make pendubot reference",
        ["ctrl_make_reference.py", "--plant", PLANT, "--params", str(PARAM_PATH), "--force"],
    )
else:
    print("Skipping reference generation.")


Make pendubot reference: python ctrl_make_reference.py --plant pendubot --params /home/jupyter-venky/underactuated_double_pendulum_package/identified_params_202_fit_20260721-231023.json --force
params: identified_params_202_fit_20260721-231023.json  sha256=30816c27d60f...

== PENDUBOT (shoulder-driven) -> pendubot_reference_202.npz ==

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

  ✗ pendubot solve failed: RuntimeError: Error in Opti::solve [OptiNode] at .../casadi/core/optistack.cpp:217:
.../casadi/core/optistack_internal.cpp:1338: Assertion "return_success(accept_limit)" failed:
Solver failed. You may use opti.debug.value to investigate the latest

RuntimeError: Make pendubot reference failed with exit code 1

## Run Hardware Control

This starts a CloudPendulum experiment. Keep `RUN_HARDWARE=False` until the cell, token, and reference are confirmed.

In [ ]:
if RUN_HARDWARE:
    args = ["ctrl_run_pendubot.py", "--params", str(PARAM_PATH), "--t-sim", str(T_SIM)]
    if not RECORD:
        args.append("--no-record")
    if ALLOW_IPOPT_FALLBACK:
        args.append("--allow-ipopt-fallback")
    if ALLOW_LEGACY_REF:
        args.append("--allow-legacy-ref")
    run_step("Run pendubot hardware control", args)
else:
    print("RUN_HARDWARE=False; hardware control not started.")

## Latest Logs

In [ ]:
cfg = load_config()
root = data_root(cfg)
cell = cfg["cell_id"]
logs = sorted(root.glob(f"ctrlrun_{PLANT}_{cell}_*.npz"), key=lambda p: p.stat().st_mtime)
if logs:
    print(f"Latest log: {logs[-1].name}")
else:
    print("No hardware log yet.")

refs = sorted(root.glob(f"{PLANT}_reference_{cell}.npz"), key=lambda p: p.stat().st_mtime)
if refs:
    print(f"Reference: {refs[-1].name}")